# Extension Desk: run the agent evals on Colab
Set the runtime first: Runtime > Change runtime type > T4 GPU. Then run the cells top to bottom.
Nothing below is pre-filled. Whatever prints is the result, including failures.

In [ ]:
!nvidia-smi | head -15
!node --version && npx --version

In [ ]:
!apt-get -y -qq install pciutils zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess, time
subprocess.Popen(['ollama', 'serve'], stdout=open('/tmp/ollama.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(8)
!ollama pull qwen2.5:7b-instruct
!ollama list

In [ ]:
REPO = 'https://github.com/ukpo217and-code/extension-desk.git'  # repo must be public, or put a token in this URL
!rm -rf extension-desk && git clone -q $REPO
%cd extension-desk
!pip install -q -e '.[agent]'
!pip show pydantic-ai-slim | head -3

In [ ]:
!PYTHONPATH=src:. python -m unittest discover -s tests -t . 2>&1 | tail -5

In [ ]:
!rm -f data/audit.db
!python -m extension_desk.cli demo | tail -25

In [ ]:
# One real agent run on one cluster. If this crashes, read the traceback: the most likely cause is a
# pydantic-ai API difference in agent/run_agent.py (it has never been executed end to end).
!rm -f data/audit.db
t0 = time.time()
!python agent/run_agent.py C-KOTON
print('seconds:', round(time.time() - t0))
!python -m extension_desk.cli pending

In [ ]:
# The 8 eval tasks, 3 runs each. Slow on a T4: expect tens of minutes.
t0 = time.time()
!python -m evals.run_evals 3
print('total minutes:', round((time.time() - t0) / 60, 1))

In [ ]:
import json
r = json.load(open('evals/results.json'))
for x in r:
    if not x['passed']:
        print(x['task'], 'run', x['run'], x['failures'])
        print('   final text:', x['final_text'][:400].replace(chr(10), ' '))

In [ ]:
from collections import defaultdict
agg = defaultdict(list)
for x in r:
    agg[x['task']].append(x['passed'])
print('model: qwen2.5:7b-instruct')
for k, v in agg.items():
    print(k, str(sum(v)) + '/' + str(len(v)))
# Copy everything printed by the last two cells and paste it into the chat.